In [1]:
import torch
from dinosaw.helpers import ModelTypes, model_names, get_models,get_features, add_custom_font
from dinosaw.utils import do_2D_pca

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = True

/home/pawlo/miniforge3/envs/minimal_dinosaw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv3', 'alibi_dv2_coco')
models = get_models(enabled_models, "../../trained_models", DEVICE, half, conf_path='../../dinov3')

In [3]:
SF = 1
img_fname = "wmg_si_c.png"
# img_fname = "default_image.jpg"

interpolation = Image.LANCZOS

_img = Image.open(f"../images/{img_fname}").convert("RGB")
_img = _img.resize((int(SF * _img.width), int(SF * _img.height)), interpolation)

In [4]:
features = {}
features_reduced = {model_key: {} for model_key in enabled_models}
for model_key in enabled_models:
    model = models[model_key]
    feats = get_features(model, _img, False, False, device=DEVICE, to_half=half)
    features[model_key] = feats

In [5]:
i = 0
for model_key in enabled_models:
    for pre_norm in ["none", "std", "minmax"]:
        feat = features[model_key]
        features_reduced[model_key][pre_norm] = do_2D_pca(feat.copy(), 9, pre_norm=None if pre_norm == "none" else pre_norm, post_norm='minmax')[:, :, i*3:i*3+3]

In [6]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [7]:
%%capture
H,W = 4,7
FS = 26
FLIP = False
fig, axs = plt.subplots(nrows=1+ len(enabled_models), ncols=3, figsize=(3*W, (1+len(enabled_models)) * H))
axs = axs.ravel()
add_custom_font("resources/fonts")
axs[0].imshow(_img)
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[2].set_axis_off()
for i, model_key in enumerate(enabled_models):
    for j, pre_norm in enumerate(["none", "std", "minmax"]):
        feats_red = features_reduced[model_key][pre_norm]
        print(i, j)
        ax = axs[3*i+3+j]
        hide_axes(ax)
        ax.imshow(np.flip(feats_red, axis=2)) if FLIP else ax.imshow(feats_red)
        if j == 0:
            ax.set_ylabel(model_names[model_key], fontsize=FS, fontweight = "bold" if "alibi" in model_key else None)
        if i == 0:
            ax.set_title(pre_norm, fontsize = FS)
plt.tight_layout()
fig.savefig("saved/S05_feature_scaling.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})